# 04. PFN: 1D GP 회귀 재현

세션 1(노트북 01~03)에서 본 GP prior, prior-fitting NLL, set을 처리하는 Transformer encoder, bar distribution을 하나로 엮어 PFN을 학습하고, 학습된 PFN이 노트북 03 §8의 analytic GP posterior(정답)를 forward pass 한 번으로 재현하는지 확인합니다.

GP를 재현 대상으로 고른 이유는 posterior가 닫힌형으로 정확히 계산돼서 PFN의 예측을 정답과 직접 대조할 수 있기 때문입니다. 하이퍼파라미터는 논문(§5.1, 부록 F.1) 값인 RBF lengthscale 0.6, variance 1.0, 관측 노이즈 1e-4로 고정합니다.

논문 크기의 모델은 H100 한 장에서 `scripts/train_pfn.py`로 학습했습니다. 이 노트북은 그 run을 MLflow에서 불러와 정답과 비교하고, 마지막에 같은 학습 루프를 작은 모델로 직접 돌려 봅니다.

다루는 흐름:

1. 정답: analytic GP posterior
2. GP prior 샘플러: PFN이 학습하는 분포
3. bar distribution: 회귀 출력 표현
4. PFN 모델
5. 학습을 어떻게 돌렸나
6. 학습 곡선
7. 검증: PFN 예측을 정답 posterior에 겹치기
8. 작은 모델로 직접 학습해 보기
9. 정리

## `minipfn` 불러오기

다른 노트북과 같은 import shim입니다. 이 노트북은 `notebooks/01_pfn/` 아래에 있으므로 저장소 루트는 두 단계 위입니다.

In [ ]:
try:
    import minipfn
except ImportError:
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
    import minipfn

In [ ]:
import os
import time
from dataclasses import asdict

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv

from minipfn import (
    PFNConfig,
    analytic_gp_posterior,
    build_criterion,
    build_model,
    build_prior_sampler,
    evaluate_against_gp,
    get_device,
    load_logged_model,
    set_seed,
    train_pfn,
)

# 저장된 출력에 artifact 다운로드 진행 막대가 남지 않게 한다
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")

# Databricks 밖(로컬)에서는 저장소 루트 .env.local의 CLI 프로필로 워크스페이스 MLflow에 접속한다.
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    load_dotenv(find_dotenv(".env.local", usecwd=True))
    mlflow.set_tracking_uri("databricks")

# H100 학습 run (paper config, 10,000 step). experiment: $MLFLOW_EXPERIMENT_DIR/pfn-gp-reproduction
RUN_ID = "bce611b854a74121bd1af60a00340638"

PAPER = PFNConfig.paper()
device = get_device()
if device.type == "mps":  # 로컬 MPS는 일부 연산 미지원 → CPU
    device = torch.device("cpu")
print("device:", device)

## 1. 정답: analytic GP posterior

노트북 03 §8과 같은 설정으로 정답을 계산합니다. context 관측 `D = (X_tr, y_tr)` 5개가 주어졌을 때의 GP posterior 평균과 분산이며, 7절에서 학습된 PFN의 예측을 이 곡선에 겹쳐 재현 여부를 판정합니다.

In [ ]:
X_grid = np.linspace(-5, 5, 200)
X_tr = np.array([-4.0, -2.5, -0.5, 1.0, 3.5])
set_seed(0)
y_tr = np.sin(X_tr) + 1e-4 * np.random.randn(X_tr.size)  # 생성 노이즈 = 가정한 노이즈

mu, cov = analytic_gp_posterior(X_tr, y_tr, X_grid)  # 기본값 ls=0.6, var=1.0, noise=1e-4
sd = np.sqrt(np.clip(np.diag(cov), 0, None))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.fill_between(X_grid, mu - 2 * sd, mu + 2 * sd, color="C0", alpha=0.2, label="analytic ±2σ")
ax.plot(X_grid, mu, color="C0", lw=2, label="analytic posterior mean")
ax.scatter(X_tr, y_tr, c="black", s=35, zorder=5, label="context D")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Reproduction target: analytic GP posterior (ls=0.6, noise=1e-4)")
ax.legend(loc="upper left", fontsize=8)
plt.show()

## 2. GP prior 샘플러: PFN이 학습하는 분포

`GPPriorSampler`는 학습 step마다 데이터셋을 batch로 뽑습니다. 데이터셋 하나는 입력 `x ~ U(-5, 5)`와, 그 x에서 zero-mean GP로 뽑은 라벨 `y ~ N(0, K(x, x) + noise²I)`로 이뤄집니다.

아래 함수 샘플이 PFN 학습 데이터의 원천입니다. lengthscale이 0.6으로 짧아서 함수가 자주 굽이칩니다.

In [ ]:
prior_sampler = build_prior_sampler(PAPER)
set_seed(1)
xs, ys = prior_sampler.sample_batch(batch_size=6, seq_len=200, device="cpu")
order = torch.argsort(xs[..., 0], dim=1)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for i in range(6):
    xi = xs[i, order[i], 0].numpy()
    yi = ys[i, order[i]].numpy()
    ax.plot(xi, yi, lw=1.0, alpha=0.8)
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("GP prior function samples (source of PFN training data)")
plt.show()

## 3. bar distribution: 회귀 출력 표현

PFN은 query마다 bucket 위의 범주분포(softmax)를 내고, bucket 안을 균일 밀도로 보아 계단형 예측분포를 만듭니다. 학습에 쓰는 `FullSupportBarDistribution`은 양 끝 bucket을 half-normal tail로 바꿔서 `[-4, 4]` 밖의 y에도 유한한 NLL을 줍니다.

아래는 무작위 logits 하나가 만드는 예측 밀도입니다.

In [ ]:
criterion = build_criterion(PAPER)
set_seed(2)
demo_logits = torch.randn(PAPER.num_buckets) * 1.5
grid = torch.linspace(-5, 5, 500)
dens = criterion.pdf(demo_logits, grid).numpy()

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(grid.numpy(), dens, color="C0")
ax.fill_between(grid.numpy(), dens, color="C0", alpha=0.25)
ax.set_xlabel("y")
ax.set_ylabel("density")
ax.set_title(f"A predictive density from bar distribution (B={PAPER.num_buckets})")
plt.show()
print("mean:", criterion.mean(demo_logits).item(), "| std:", criterion.std(demo_logits).item())

## 4. PFN 모델: set을 처리하는 Transformer encoder

각 (x, y)가 토큰 하나입니다. context 토큰에는 실제 y를 넣고, query 토큰에는 y 대신 mask 토큰을 넣습니다.

positional encoding이 없어서 토큰 순서가 바뀌어도 결과가 같습니다. attention mask로 query는 context에만 attend하게 해서(다른 query와 자기 정답은 보지 않음) 여러 query를 한 번에 독립적으로 예측합니다. 출력 head는 query마다 bar distribution logits를 냅니다.

아래는 H100에서 학습한 논문 크기(paper config) 모델의 구조입니다.

In [ ]:
model = build_model(PAPER)
print("parameters:", f"{model.num_parameters():,}")
print(model.encoder)

## 5. 학습을 어떻게 돌렸나

학습은 논문 Algorithm 1을 그대로 반복합니다. 매 step GP prior에서 데이터셋 batch를 새로 뽑고, context 개수를 무작위로 정해 context와 query로 나눈 뒤, query 라벨의 NLL을 최소화합니다. 실제 데이터는 쓰지 않습니다.

설정은 `PFNConfig.paper()`입니다. 모델 크기와 step 수는 논문 값이고, 논문에 없는 값은 이 저장소의 기본값을 썼습니다.

| 항목 | 값 | 출처 |
|---|---|---|
| 모델 | 6층, emsize 512, head 4, FFN 1024, dropout 0 | 논문 |
| step | 10,000 (100 epoch × 100 step) | 논문 |
| 출력 | bar distribution, bucket 100개, 범위 [-4, 4] | 이 저장소 |
| 데이터셋 하나 | 점 200개, context 5~190개(나머지는 query) | 이 저장소 |
| batch | 데이터셋 128개 | 이 저장소 |
| 최적화 | Adam, lr 3e-4, warmup 1,000 step + cosine | 이 저장소 |

실행 파일은 저장소 루트의 `runs/`와 `scripts/`에 있습니다.

| 파일 | 하는 일 |
|---|---|
| `runs/train_pfn.yaml` | AI Runtime workload 설정입니다. 환경 AI v6, GPU 한 장(기본 A10)을 씁니다. |
| `scripts/train_pfn.py` | 인자와 환경변수를 `PFNConfig`로 바꾸고 `train_pfn`을 부른 뒤, `evaluate_against_gp`로 정답과 비교한 지표와 그림을 MLflow run에 기록합니다. |

스크립트가 하는 일은 인자 처리와 MLflow 기록뿐이고, 학습과 평가 로직은 이 노트북이 import하는 `src/minipfn/`의 함수를 그대로 씁니다. 이 노트북이 불러오는 run은 저장소 루트에서 다음 명령으로 돌렸습니다. yaml 기본값은 A10이라 GPU 종류만 H100으로 바꿨습니다.

```bash
databricks air run -f runs/train_pfn.yaml --watch --override mlflow_experiment_directory=$MLFLOW_EXPERIMENT_DIR \
  --override compute.accelerator_type=GPU_1xH100 \
  --override env_variables.PFN_CONFIG_NAME=paper --override env_variables.PFN_STEPS=10000 \
  --override mlflow_run_name=pfn-paper-10000steps-h100 --override timeout_minutes=90
```

MLflow에 기록된 config가 `PFNConfig.paper()`와 같은지 확인합니다.

In [ ]:
run = mlflow.get_run(RUN_ID)
params = run.data.params
preset = asdict(PAPER)
mismatched = [k for k, v in preset.items() if params.get(k) != str(v)]

pd.DataFrame(
    [
        {
            "MLflow run name": run.info.run_name,
            "job_run_id": run.data.tags.get("job_run_id"),
            "device": params["device"],
            "steps": params["steps"],
            "파라미터 수": params["n_parameters"],
            "preset과 다른 항목": ", ".join(mismatched) or "없음",
        }
    ]
)

## 6. 학습 곡선

run에 기록된 metric을 불러와 그립니다. `train_nll`은 10 step마다 그 step에 새로 뽑은 batch의 query NLL이고, `val_nll`은 학습 시작 전에 따로 뽑아 고정해 둔 prior batch의 query NLL입니다. 매 step 새 데이터셋을 뽑으므로 학습 데이터를 다시 보는 일이 없어서, 두 곡선이 벌어지는 과적합은 생기지 않습니다.

NLL이 음수까지 내려가는 것은 관측 노이즈가 1e-4로 작기 때문입니다. context가 많고 가까이 있는 query에서는 예측분포가 아주 좁아져서 밀도가 1보다 커지고, 그러면 −log 밀도는 음수가 됩니다. 7절의 NLL은 context를 5개만 주고 계산하므로 이보다 큽니다.

In [ ]:
client = mlflow.MlflowClient()


def metric_history(key):
    history = client.get_metric_history(RUN_ID, key)
    steps = np.array([m.step for m in history])
    values = np.array([m.value for m in history])
    order = np.argsort(steps)
    return steps[order], values[order]


train_step, train_nll = metric_history("train_nll")
val_step, val_nll = metric_history("val_nll")

window = 25  # 기록 25개(= 250 step) 이동 평균
smooth = np.convolve(train_nll, np.ones(window) / window, mode="valid")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(train_step, train_nll, color="C0", alpha=0.25, lw=0.8, label="train NLL")
ax.plot(
    train_step[window // 2 : window // 2 + len(smooth)],
    smooth,
    color="C0",
    lw=2,
    label="train NLL (smoothed)",
)
ax.plot(val_step, val_nll, "o-", color="C1", ms=4, label="val NLL (fixed prior batch)")
ax.set_xlabel("step")
ax.set_ylabel("query NLL")
ax.set_title("Prior-fitting loss (paper config, H100)")
ax.legend()
plt.show()
print(f"마지막 val NLL {val_nll[-1]:.3f} (step {val_step[-1]})")

## 7. 검증: PFN 예측을 정답 posterior에 겹치기

학습이 끝난 모델(`final_model`)을 MLflow에서 불러옵니다. 1절과 같은 context D 5개와 같은 grid를 PFN에 넣으면, 다시 학습하지 않고 forward pass 한 번으로 예측분포가 나옵니다.

`evaluate_against_gp`는 학습 스크립트가 run을 마칠 때 부른 함수와 같습니다. 정답 곡선에 PFN 예측을 겹친 그림과 함께 다음 지표를 돌려줍니다.

| 지표 | 뜻 |
|---|---|
| `rmse_mean` | grid 200점에서 PFN 평균과 GP 평균의 RMSE |
| `mean_abs_dsigma` | grid 200점에서 PFN 표준편차와 GP 표준편차 차이의 절댓값 평균 |
| `pfn_nll`, `gp_nll` | query 40점(y = sin x)에서 PFN과 GP의 평균 NLL |
| `nll_gap` | `pfn_nll − gp_nll` |
| `zero_ctx_abs_mean`, `zero_ctx_std` | context를 하나도 주지 않았을 때, grid 점마다 낸 예측 평균의 절댓값 평균과 예측 표준편차의 평균 (참고값) |

모델을 불러올 때 나오는 PyTorch 버전 경고는 저장한 쪽(GPU, `+cu130` 빌드)과 불러오는 쪽의 빌드 표기가 달라서 나옵니다. 버전은 2.11.0으로 같아서 무시해도 됩니다.

In [ ]:
loaded = load_logged_model(f"runs:/{RUN_ID}/final_model", map_location=device)
paper_model, paper_criterion = loaded["model"], loaded["criterion"]

paper_metrics, fig = evaluate_against_gp(
    paper_model, paper_criterion, title="PFN (paper config, H100) vs analytic GP posterior"
)
plt.show()
pd.Series(paper_metrics).round(4).to_frame("paper (H100)")

재현 판정은 앞의 두 가지로 하고, 세 번째는 참고로 봅니다.

- **평균과 밴드가 겹치는가**: 그림에서 PFN 평균(빨간 점선)이 정답 평균을 따라가고, ±2σ 밴드가 context 근처에서 좁아졌다가 멀어지면 넓어지는 모양까지 같으면 재현입니다.
- **NLL이 GP와 비슷한가**: GP posterior는 prior에서 뽑은 데이터셋들에 대해 평균을 내면 NLL이 가장 낮은(Bayes-optimal) 예측입니다. 그래서 `nll_gap`이 0에 가까우면 PFN이 그 최적 예측에 가깝다는 뜻입니다. 다만 여기 query는 sin 함수 하나라서, PFN NLL이 GP보다 조금 낮게 나와도 이상하지 않습니다. GP NLL은 이 함수 하나에 대한 하한이 아닙니다.
- **context가 없을 때 (참고)**: context를 하나도 주지 않으면 정확한 답은 GP prior 자체입니다. 평균은 0, 표준편차는 √(variance + noise²) ≈ 1.0입니다. 다만 학습할 때 context는 최소 5개(`min_context`)부터 뽑아서, 0개는 모델이 한 번도 본 적 없는 입력입니다. 그래서 이 값은 학습 범위 밖에서 얼마나 그럴듯하게 외삽하는지를 보는 참고값이고, 재현 판정에는 쓰지 않습니다. 평균은 점마다 부호가 달라 서로 지워질 수 있어서 절댓값으로 평균을 냅니다.

## 8. 작은 모델로 직접 학습해 보기

같은 학습 루프(`train_pfn`)를 이 노트북에서 작은 모델로 돌려 봅니다. `PFNConfig.small()`은 2층, emsize 32, batch 16, 1,200 step이라 CPU에서 10초가 안 걸립니다.

seed를 정한 직후에 모델을 만드는 순서는 `scripts/train_pfn.py`와 같습니다. 그래서 초기 가중치가 `--config-name small`로 돌린 스크립트와 같습니다. 이 셀은 MLflow에 기록하지 않습니다.

In [ ]:
SMALL = PFNConfig.small()
set_seed(SMALL.seed)
small_model = build_model(SMALL)
small_criterion = build_criterion(SMALL)
small_prior_sampler = build_prior_sampler(SMALL)
print("parameters:", f"{small_model.num_parameters():,}")

start = time.perf_counter()
history = train_pfn(
    small_model, small_prior_sampler, small_criterion, SMALL, device=device, progress=False
)
print(f"학습 {SMALL.steps:,} step: {time.perf_counter() - start:.1f}초 ({device})")
losses = np.array(history["losses"])

window = 25
smooth = np.convolve(losses, np.ones(window) / window, mode="valid")
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(losses, color="C0", alpha=0.3, lw=0.8, label="train NLL")
ax.plot(np.arange(len(smooth)) + window // 2, smooth, color="C0", lw=2, label="smoothed")
ax.set_xlabel("step")
ax.set_ylabel("query NLL")
ax.set_title("Prior-fitting loss (small config, this notebook)")
ax.legend()
plt.show()
print(f"처음 20 step 평균 {losses[:20].mean():.3f} → 마지막 50 step 평균 {losses[-50:].mean():.3f}")

작은 모델도 7절과 같은 함수로 정답과 비교하고, 두 모델의 지표를 나란히 봅니다.

In [ ]:
small_metrics, fig = evaluate_against_gp(
    small_model, small_criterion, title="PFN (small config, this notebook) vs analytic GP posterior"
)
plt.show()
pd.DataFrame({"small (이 노트북)": small_metrics, "paper (H100)": paper_metrics}).round(4)

## 9. 정리

| | small (이 노트북) | paper config (H100) |
|---|---|---|
| 파라미터 수 | 28,868 | 12,670,564 |
| 학습 | 1,200 step × batch 16, CPU 10초 미만 | 10,000 step × batch 128, H100 한 장 약 16분 |
| RMSE(mean) | 0.136 | 0.068 |
| mean\|Δσ\| | 0.128 | 0.027 |
| NLL 차이 (PFN − GP) | +0.144 | +0.007 |
| context 0개일 때 \|평균\| / 표준편차 (참고, 정답 0 / 1.0) | 0.50 / 1.81 | 0.07 / 1.02 |

논문 크기 모델은 정답 posterior와 평균, ±2σ 밴드가 거의 겹치고, NLL도 GP와 0.007 차이입니다. 실제 데이터 없이 prior에서 뽑은 데이터셋만으로 학습한 Transformer가 forward pass 한 번으로 GP posterior를 근사한다는 논문의 주장을 이 setup에서 확인했습니다.

작은 모델도 평균은 대체로 따라가지만 불확실성이 맞지 않습니다. RMSE(mean)는 paper 모델의 2배인데 표준편차 차이(mean|Δσ|)는 약 5배이고, NLL도 GP보다 0.144 높습니다. 모델 크기와 학습량이 모자라면 평균보다 불확실성에서 먼저 드러납니다.

한계도 있습니다.

- **평가는 함수 하나 기준입니다.** sin 함수와 context 5개라는 고정된 예 하나로 비교했습니다. prior에서 뽑은 데이터셋 여러 개로 평균 NLL을 비교하지는 않았습니다.
- **context 0개는 학습 범위 밖입니다.** 학습은 context를 5개부터 뽑아서, context가 없을 때의 값은 참고로만 봤습니다.
- **GP 하이퍼파라미터를 고정했습니다.** 논문처럼 하이퍼파라미터에도 prior를 두고 뽑는 실험은 하지 않았습니다.
- **batch, lr, bucket 수는 논문에 없어서 이 저장소의 기본값을 썼습니다.**